# AromaLLM: Chemical Odor Prediction

### 1. Setup and Environment

In [1]:
!pip install -q transformers accelerate bitsandbytes peft sentencepiece einops safetensors
!git clone https://github.com/SanchesIH/aromaLLM.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.0 MB/s eta 0:00:00
Cloning into 'aromaLLM'...
remote: Enumerating objects: 93, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 93 (delta 22), reused 83 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (93/93), 31.06 MiB | 18.25 MiB/s, done.
Resolving deltas: 100% (22/22), done.


### 2. Load Base Model and Tokenizer

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

model_id = "OpenDFM/ChemDFM-v1.5-8B"
lora_path = "/content/aromaLLM/chemdfm_finetuned"

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/865 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.1k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/439 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

### 3. Apply LoRA Weights

In [3]:
model = PeftModel.from_pretrained(base_model, lora_path)
model.eval()
print("Model and LoRA weights loaded successfully!")

Model and LoRA weights loaded successfully!


### 4. Inference Test

In [4]:
smiles = "CCCCSSCCCC"
prompt = f"[Round 0]\nHuman: What are the primary odors and sub-odors of this molecule: {smiles}?\nAssistant:"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False
    )

response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f"SMILES: {smiles}")
print(f"Prediction: {response}")

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


SMILES: CCCCSSCCCC
Prediction:  The primary odors present in this molecule are: Septic, Vegetables, Nature, Sulp, Hydrocar, Cabbage, Chemical, Alliaceous, Marshy, Sulphur, Sickening. The sub-odors are: Sulphurous, Onion, Alliaceous, Garlic. The roundic sub-odors are: Sulphurous, Garlic. The non-roundic sub-odors are: Garlic. The other sub-odors are: Sulphurous, Alliaceous, Garlic, Onion. The sub-odorsRoundic are: Sulphurous, Garlic. The sub-


### Applying llmSHAP

In [5]:
!pip install "llmshap[all]"

In [6]:
from llmSHAP.llm.llm_interface import LLMInterface
import torch

class ChemDFMInterface(LLMInterface):
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer

    def generate(
        self,
        prompt,
        tools=None,
        images=None,
    ) -> str:

        # Convert prompt into text
        if isinstance(prompt, list):
            prompt_text = ""

            for msg in prompt:
                role = msg.get("role", "")
                content = msg.get("content", "")

                if role == "system":
                    prompt_text += f"{content}\n"

                elif role == "user":
                    prompt_text += f"[Round 0]\nHuman: {content}\nAssistant:"

        else:
            prompt_text = str(prompt)

        inputs = self.tokenizer(
            prompt_text,
            return_tensors="pt"
        ).to("cuda")

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                repetition_penalty=1.15,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id
            )

        generated = outputs[0][inputs["input_ids"].shape[1]:]

        return self.tokenizer.decode(
            generated,
            skip_special_tokens=True
        ).strip()

In [7]:
chem_model = ChemDFMInterface(model, tokenizer)

In [14]:
from llmSHAP import DataHandler, BasicPromptCodec, ShapleyAttribution
from llmSHAP.llm import OpenAIInterface

smiles = "CCCCSSCCCC"

molecules = {
    "mol_garlic": "CCCCSSCCCC",
    "mol_rose":   "OCC1=CC=CC=C1",
    "mol_mint":   "CC1CCC(C(C1)O)C(C)C",
    "question":   "What are the primary odors of this molecule?",
}
handler = DataHandler(molecules, permanent_keys={"question"})

result = ShapleyAttribution(
    model=chem_model,
    data_handler=handler,
    prompt_codec=BasicPromptCodec(
        system="You are an expert in molecular odor prediction."
    ),
    use_cache=True,
    num_threads=1,
).attribution()

print("\n\n### OUTPUT ###")
print(result.output)

print("\n\n### ATTRIBUTION ###")
print(result.attribution)

print("\n\n### HEATMAP ###")
print(result.render(abs_values=True, render_labels=True))

Features:   0%|          | 0/4 [00:00<?, ?it/s]

Coalitions:   0%|          | 0/4 [00:00<?, ?it/s]

Coalitions:   0%|          | 0/4 [00:00<?, ?it/s]

Coalitions:   0%|          | 0/4 [00:00<?, ?it/s]

Time (3 features): 139.88 seconds.


### OUTPUT ###
The primary odors present in this molecule are: Vegetation, Fruity, Aromatic, Nature, Woody, Resinous, Grassy, Sweet, Fragrant.


### ATTRIBUTION ###
{'mol_garlic': {'value': 'CCCCSSCCCC', 'score': 0.1465247899373722}, 'mol_rose': {'value': 'OCC1=CC=CC=C1', 'score': 0.18295665220596016}, 'mol_mint': {'value': 'CC1CCC(C(C1)O)C(C)C', 'score': 0.20036141061223506}, 'question': {'value': 'What are the primary odors of this molecule?', 'score': 0}}


### HEATMAP ###
 mol_garlic   mol_rose   mol_mint   question 
